In [58]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('./data/pet_adoption_data.csv')

df = df.drop(columns=['PetID'])


def symulacja_budzetu(y_true, y_proba, budzet=200, koszt_reklamy=10):
    """
    Funkcja oceniająca kampanię na podstawie przewidzianych prawdopodobieństw.
    
    Parametry:
    y_true - rzeczywiste etykiety (1 = szansa na adopcję)
    y_proba - prawdopodobieństwa klasy 1 wyznaczone przez model
    budzet - całkowity budżet (zgodnie z zadaniem 200$)
    koszt_reklamy - koszt jednej reklamy (10$)
    
    Zwraca:
    Oczekiwaną, łączną liczbę zaadoptowanych zwierząt.

    
    """
    if y_proba.ndim == 2:
        y_proba = y_proba[:, 1]

    max_reklam = budzet // koszt_reklamy 
    
    wyniki = pd.DataFrame({'y_true': y_true, 'proba': y_proba})
    wyniki_posortowane = wyniki.sort_values(by='proba', ascending=False)
    
    wybrane = wyniki_posortowane.head(max_reklam)
    
    tp = (wybrane['y_true'] == 1).sum()
    fp = (wybrane['y_true'] == 0).sum()
    
    oczekiwana_liczba_adopcji = (tp * 0.70) + (fp * 0.10)
    
    return oczekiwana_liczba_adopcji
y = df['AdoptionLikelihood']
X = df.drop('AdoptionLikelihood', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

zmienne_kategoryczne = ['PetType', 'Breed', 'Color', 'Size']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), zmienne_kategoryczne)
    ],
    remainder='passthrough'
)


In [60]:
from sklearn.metrics import make_scorer

def moja_metryka(y_true, y_proba, **kwargs):
    return symulacja_budzetu(y_true, y_proba)

scorer = make_scorer(moja_metryka, needs_proba=True)

In [61]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import ConfusionMatrixDisplay

clf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])
clf.fit(X_train, y_train)


In [62]:
param_grid = {
    'model__max_depth': [2, 3, 5, 8, 10, None],
    'model__min_samples_leaf': [1, 5, 10, 20],
    'model__class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    param_grid,
    cv=cv,
    scoring=scorer,
    n_jobs=-1
)

grid.fit(X_train, y_train)
y_pred = grid.best_estimator_.predict(X_test)
print(grid.best_params_)
print(grid.best_score_)


In [63]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 8))
feature_names = grid.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
plot_tree(
    grid.best_estimator_.named_steps['model'],  
    max_depth=3,   
    filled=True,         
    feature_names=feature_names,
    class_names=['Nie adoptowany', 'Adoptowany'],
    rounded=True
)
plt.show()


In [64]:
symulacja_budzetu(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1])

In [65]:
y_proba_base = clf.predict_proba(X_test)[:, 1]
y_proba_tuned = grid.best_estimator_.predict_proba(X_test)[:, 1]

print('Bazowe drzewo:', symulacja_budzetu(y_test, y_proba_base))
print('Po tuningu:', symulacja_budzetu(y_test, y_proba_tuned))
print('Baseline losowy:', baseline)

In [66]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    grid.best_estimator_.predict(X_test),
    display_labels=['Nie adoptowany', 'Adoptowany'],
    cmap='Blues'
)
plt.title('Macierz błędów')
plt.show()

In [67]:
X_train_transformed = preprocessor.fit_transform(X_train)
path = grid.best_estimator_.named_steps['model'].cost_complexity_pruning_path(X_train_transformed, y_train)
alphas = path.ccp_alphas[:-1]  # ostatnia to korzeń, pomijamy
scores = []
for alpha in alphas[::max(1, len(alphas)//20)]:  # co kilkanaście żeby nie było ich za dużo
    clf_pruned = Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(ccp_alpha=alpha, random_state=42))
    ])
    score = cross_val_score(clf_pruned, X_train, y_train, cv=cv, scoring=scorer).mean()
    scores.append((alpha, score))

a, s = zip(*scores)
plt.plot(a, s, marker='o')
plt.xlabel('ccp_alpha')
plt.ylabel('Symulacja budżetu')
plt.show()


In [68]:
import numpy as np
best_alpha = a[np.argmax(s)]
clf_pruned = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42))
])
clf_pruned.fit(X_train, y_train)
y_pred_pruned = clf_pruned.predict(X_test)
print(symulacja_budzetu(y_test, clf_pruned.predict_proba(X_test)[:, 1]))


In [69]:
results = pd.DataFrame(grid.cv_results_)
depth_results = results.groupby('param_model__max_depth')['mean_test_score'].max().reset_index()
depth_results['param_model__max_depth'] = depth_results['param_model__max_depth'].astype(str)

plt.figure(figsize=(8, 4))
plt.plot(depth_results['param_model__max_depth'], depth_results['mean_test_score'], marker='o')
plt.xlabel('max_depth')
plt.ylabel('Symulacja budżetu (CV)')
plt.title('Wpływ głębokości drzewa na wynik')
plt.show()


In [70]:
import joblib

# Zapisanie wybranego, wytrenowanego modelu
joblib.dump(grid.best_estimator_, 'Tree_shelter_model.pkl')
print('Zapisano model do pliku Tree_shelter_model.pkl')